Imports:

In [ ]:
%pip install -q requests pandas numpy statsmodels openpyxl

In [ ]:
import requests
import pandas as pd
import statsmodels.api as sm

Daten laden und vorbereiten:

In [ ]:
url = "https://api.kbstats.de/api/v1/players" # inoffizielle API
response = requests.get(url)
response.raise_for_status() # Fehler werfen, falls Request schiefgeht
players = response.json()
# In DataFrame umwandeln
df = pd.DataFrame(players)
# nur relevante Spalten auswählen
df = df[["name", "marketValue", "totalPoints", "position", "trend"]]
# Daten bereinigen (nur Spieler mit sinnvollen Werten)
df = df.dropna(subset=["marketValue", "totalPoints", "trend", "position"])
df = df[(df["marketValue"] > 0) & (df["totalPoints"] > 0)]

Weicht der MW eines Spielers
systematisch von einem „fairen“
wertbasierten MW ab?

In [ ]:
# Positionen in Labels
pos_map = {1: "Torwart", 2: "Verteidiger", 3: "Mittelfeld", 4: "Angreifer"}
df["pos_label"] = df["position"].map(pos_map)
# Dummies erstellen
dummies = pd.get_dummies(df["pos_label"], dtype=int)
# Torwart als Referenz: TW-Dummy NICHT ins Modell!
dummies = dummies.drop(columns=["Torwart"])
# Interaktionsvariablen definieren
dummies["Int_points_V"] = df["totalPoints"] * dummies["Verteidiger"]
dummies["Int_points_M"] = df["totalPoints"] * dummies["Mittelfeld"]
dummies["Int_points_A"] = df["totalPoints"] * dummies["Angreifer"]
# Modell erstellen
df2 = pd.concat([df, dummies], axis=1)
X = df2[[
    "totalPoints",
    "Verteidiger", "Mittelfeld", "Angreifer",
    "Int_points_V", "Int_points_M", "Int_points_A"
]]
X = sm.add_constant(X)
y = df2["marketValue"]
model = sm.OLS(y, X).fit()
print(model.summary())
# Vorhersagen berechnen
df["predMarketValue"] = model.predict(X).values
# Differenz -> positiv = "zu teuer", negativ = "zu billig"
df["diff"] = df["marketValue"] - df["predMarketValue"]

In [ ]:
# gewünschte Ausgabe für EXCEL
out = df[["name", "position", "marketValue", "predMarketValue", "diff"]].copy()
# formatieren
out = out.rename(columns={
    "marketValue": "marktwert",
    "predMarketValue": "vorhergesagter_marktwert",
    "diff": "differenz"
})
out["marktwert"] = out["marktwert"].round(0)
out["vorhergesagter_marktwert"] = out["vorhergesagter_marktwert"].round(0)
out["differenz"] = out["differenz"].round(0)
# Excel erstellen
out.to_excel("kickbase_output.xlsx", index=False)